# SPTSSB Haplotype Analysis - Test 2
Since the first test using Mary Makarious pipeline didn't work (cannot handle more than ~10 variants and there are 24 variants of interest), I decided to try a different approach.

LDhap online tool already prioritized 4 clear haplotypes. What we need to know now are the frequencies of each haplotype in PD cases and controls separately.

In [ ]:
## Load PLINK data
library(haplo.stats)

haplo_data <- haplo.stats::read.pedfile("sptssb_phased_filtered_3.ped", "sptssb_phased_filtered_3.map")
#geno <- haplo_data$geno
#locus.label <- haplo_data$locus.label

In [ ]:
## Extract phenotype from .ped file
ped <- read.table("yourfile.ped", stringsAsFactors = FALSE)
pheno <- ped[,6]

In [ ]:
## Define known haplotypes
known_haps <- list(
  hap1 = c("A","T","C","G"),
  hap2 = c("G","T","C","A"),
  hap3 = c("A","C","T","G")
)

In [ ]:
## Assign known haplotypes to individuals
n <- nrow(geno)
num_snps <- length(locus.label)
hap_counts <- matrix(0, nrow=n, ncol=length(known_haps))
colnames(hap_counts) <- names(known_haps)

for (i in 1:n) {
  hap1 <- geno[i, seq(1, 2*num_snps, by=2)]   # allele1
  hap2 <- geno[i, seq(2, 2*num_snps, by=2)]   # allele2
  
  for (j in seq_along(known_haps)) {
    hap <- known_haps[[j]]
    hap_counts[i,j] <- sum(
      all(hap1 == hap),
      all(hap2 == hap)
    )
  }
}

In [ ]:
## Summarize frequencies per phenotype
case_freqs <- colSums(hap_counts[pheno == 2, ]) / (2 * sum(pheno == 2))
ctrl_freqs <- colSums(hap_counts[pheno == 1, ]) / (2 * sum(pheno == 1))

data.frame(Haplotype = names(known_haps),
           CaseFreq = round(case_freqs, 4),
           CtrlFreq = round(ctrl_freqs, 4))

In [ ]:
## Statistics
# Chi-square test comparing carrier frequencies (e.g., hap1+ vs hap1−)
hap1_carrier <- hap_counts[, "hap1"] > 0
table(pheno, hap1_carrier)
chisq.test(table(pheno, hap1_carrier))

# Logistic regression
glm(pheno ~ hap_counts[, "hap1"], family=binomial)